# 01 — Data Exploration: tgbl-flight Dataset

**Project**: AirRoute Intelligence — Temporal Link Prediction  
**Dataset**: tgbl-flight from the Temporal Graph Benchmark (TGB)  
**Task**: Predict future flight interactions between airport pairs

---

## What is this notebook?

This notebook explores the `tgbl-flight` dataset before building any model. The goal is to understand:
1. **What the data represents** — airports as nodes, flights as temporal interactions
2. **The scale of the data** — how many airports, flights, and time steps
3. **Temporal patterns** — how flight activity evolves over time
4. **Network structure** — degree distribution, hubs, density
5. **Node and edge features** — what information is available for each airport and flight

### Key Concepts

- **Interaction** (not "edge" in the static sense): a flight observed between two airports on a specific day. Multiple interactions can occur between the same pair.
- **Temporal Graph**: $G(t) = (V, E(t))$ where $V$ is the set of airports and $E(t)$ are the interactions observed up to time $t$.
- **Node Features**: properties of airports (type, location, continent).
- **Edge Features**: properties of flights (aircraft type code).

## 1. Dataset Source and Loading

The dataset comes from the **Temporal Graph Benchmark (TGB)**, published at NeurIPS 2023.

- **Name**: `tgbl-flight` (internal version 2)
- **Source**: OpenSky Network — crowdsourced ADS-B flight tracking data
- **Reference**: Strohmeier et al., *Crowdsourced air traffic data from the OpenSky Network 2019–2020*, ESSD, 2021
- **License**: OpenSky License
- **Package**: `py-tgb >= 2.3.0`

The raw files are:
- `tgbl-flight_edgelist_v2.csv` (2.2 GB) — all 67M interactions
- `airport_node_feat_v2.csv` (3.3 MB) — airport features
- `tgbl-flight_val_ns_v2.pkl` / `tgbl-flight_test_ns_v2.pkl` — pre-generated negative samples

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datetime import datetime, timezone

sns.set_theme(style='whitegrid', palette='muted')

CSV_PATH = os.path.join('..', 'datasets', 'tgbl-flight', 'tgbl-flight_edgelist_v2.csv')
NODE_FEAT_PATH = os.path.join('..', 'datasets', 'tgbl-flight', 'airport_node_feat_v2.csv')

## 2. Node Features — Airport Properties

Each airport has 6 features. Let's inspect them.

In [ ]:
nf = pd.read_csv(NODE_FEAT_PATH)
print(f'Node feature table: {nf.shape[0]} airports, {nf.shape[1]} columns')
print(f'Columns: {list(nf.columns)}')
nf.head(10)

In [ ]:
print('=== Column Analysis ===')
for col in nf.columns:
    n_unique = nf[col].nunique()
    n_null = nf[col].isnull().sum()
    print(f'{col}: {n_unique} unique, {n_null} nulls ({100*n_null/len(nf):.1f}%)')
    if n_unique <= 10:
        print(f'  Values: {nf[col].value_counts().to_dict()}')

### Interpretation

The node feature table contains **57,421 airports** worldwide with:
- **airport_code**: ICAO code (e.g., KORD = Chicago O'Hare, EGLL = London Heathrow)
- **type**: 7 categories — `large_airport` (616), `medium_airport` (4,537), `small_airport` (34,808), `heliport` (12,028), `closed` (4,378), `seaplane_base` (1,030), `balloonport` (24)
- **continent**: 6 continents, but **28,443 nulls (49.5%)** — North America airports have no continent label
- **iso_region**: ISO region code (e.g., US-IL, GB-ENG) — 2,776 unique regions, no nulls
- **longitude / latitude**: geographic coordinates

**Note**: Only 18,143 of these airports appear in the edge data. The CSV contains the full OurAirports database.

## 3. Edge Data — Flight Interactions

Let's look at the structure of the interaction data.

In [ ]:
sample = pd.read_csv(CSV_PATH, nrows=20)
print(f'Columns: {list(sample.columns)}')
print(f'\nDtypes:')
print(sample.dtypes)
sample

### Column Meanings

| Column | Type | Meaning |
|--------|------|--------|
| `timestamp` | int64 | UNIX epoch (seconds) — one value per day |
| `src` | string | Source airport ICAO code |
| `dst` | string | Destination airport ICAO code |
| `callsign` | string | Flight callsign (e.g., HVN19, CCA839) |
| `typecode` | string | Aircraft type code (e.g., A332, B788) — 40.7% null |

Each row is a **flight interaction**: aircraft of type `typecode` with callsign `callsign` flew from `src` to `dst` on the day identified by `timestamp`.

**Important distinction**:
- This is a **flight observation**, not a confirmed commercial route
- Self-loops exist (src == dst) — these are likely training flights or touch-and-go operations
- The data is **crowdsourced** from ADS-B receivers, so coverage varies by region

## 4. Global Statistics

Due to the large file size (67M rows, 2.2 GB), we read the data in chunks.

In [ ]:
# Read in chunks to compute source/destination counts
src_counts = Counter()
dst_counts = Counter()
n_total = 0

for chunk in pd.read_csv(CSV_PATH, chunksize=10_000_000, usecols=['src', 'dst']):
    n_total += len(chunk)
    src_counts.update(chunk['src'].value_counts().to_dict())
    dst_counts.update(chunk['dst'].value_counts().to_dict())
    print(f'  Processed {n_total:,} rows')

all_nodes = set(src_counts.keys()) | set(dst_counts.keys())
total_deg = {n: src_counts.get(n, 0) + dst_counts.get(n, 0) for n in all_nodes}

print(f'\n=== SUMMARY ===')
print(f'Total interactions: {n_total:,}')
print(f'Unique sources: {len(src_counts):,}')
print(f'Unique destinations: {len(dst_counts):,}')
print(f'Unique airports (nodes): {len(all_nodes):,}')

### Key Numbers

| Metric | Value |
|--------|------:|
| Total interactions | **67,169,570** |
| Unique airports (nodes) | **18,143** |
| Unique sources | 16,706 |
| Unique destinations | 17,927 |
| Period | 2019-01-01 to 2022-10-31 |
| Unique timestamps (days) | 1,385 |
| Timestamp granularity | ~1 day (86,400 seconds) |
| Unique (src, dst) pairs | ~2,309,707 |
| Directed density | 0.70% |
| Official eval metric | MRR (Mean Reciprocal Rank) |

### Interpretation

- The network is **sparse** (0.70% density) — most airport pairs never have a direct flight.
- There are **many repeated interactions**: on average ~29 interactions per unique pair, meaning many routes are regularly flown.
- The 1,437 airports that appear only as sources or destinations (not both) suggest directional patterns in the network.
- Timestamps are daily-resolution UNIX epochs.

## 5. Degree Distribution

The **degree** of a node is the total number of interactions involving it. In a directed graph:
- **Out-degree**: number of outgoing flights
- **In-degree**: number of incoming flights
- **Total degree**: out + in

This tells us how "busy" each airport is.

In [ ]:
from IPython.display import Image
Image(filename='../results/figures/01_degree_distribution.png')

### Interpretation

The degree distribution follows a **power law** (straight line on log-log scale). This is the hallmark of a **scale-free network**:
- **Most airports** have very few interactions (small regional airports)
- **A few hubs** concentrate a huge number of interactions (major international airports)

This is consistent with the **hub-and-spoke model** of air transport. It also means:
- **For link prediction**: hubs will have more positive training examples, making predictions easier for them
- **For resilience**: removing a hub has outsized impact on the network

## 6. Top 20 Airports (Hubs)

In [ ]:
Image(filename='../results/figures/02_top20_airports.png')

In [ ]:
# Top 20 with details
top20 = sorted(total_deg.items(), key=lambda x: -x[1])[:20]
active_nf = nf[nf['airport_code'].isin(all_nodes)].set_index('airport_code')

print(f'{"Airport":8s} {"Type":18s} {"Region":8s} {"Out":>10s} {"In":>10s} {"Total":>10s}')
print('-' * 70)
for code, deg in top20:
    row = active_nf.loc[code] if code in active_nf.index else None
    atype = row['type'] if row is not None else '?'
    region = row['iso_region'] if row is not None else '?'
    out = src_counts.get(code, 0)
    inp = dst_counts.get(code, 0)
    print(f'{code:8s} {atype:18s} {region:8s} {out:>10,} {inp:>10,} {deg:>10,}')

### Interpretation

The top hubs are dominated by **US airports** (KORD, KATL, KDFW, KLAX...) which reflects the heavy coverage of ADS-B receivers in North America. European hubs (EHAM, EGLL, EDDF, LFPG) also appear.

**Bias alert**: The OpenSky Network's crowdsourced nature means coverage is denser in regions with more ADS-B receivers. North American and European traffic is better captured than African or Asian traffic. This is a dataset limitation, not a reflection of actual global traffic volumes.

All top 20 are `large_airport` type, confirming that hub status correlates with airport size.

## 7. In-degree vs Out-degree

In [ ]:
Image(filename='../results/figures/05_in_vs_out_degree.png')

### Interpretation

Most airports cluster close to the $y=x$ line, meaning **in-degree ≈ out-degree**: airports that receive many flights also send many. This is physically expected — planes that land must eventually take off.

Some outliers exist:
- Airports above the line: more incoming than outgoing (possible destination-heavy airports or data collection artifacts)
- Airports below the line: more outgoing (possible base airports for training or cargo)

For modeling, this suggests the graph can be treated as **approximately symmetric** at the aggregate level, though individual daily interactions are inherently directed.

## 8. Airport Types and Continents

In [ ]:
Image(filename='../results/figures/06_airport_types_continents.png')

### Interpretation

**Airport Types**:
- 82% of active nodes are `small_airport` — most airports in the world are small
- Only 484 `large_airport` nodes exist, but they concentrate most of the traffic
- Some `closed` airports (563) appear, likely with legacy data before closure

**Continents** (among active airports with continent labels):
- Europe leads (2,702), followed by South America (1,268) and Asia (743)
- Note: ~49.5% of airports in the full feature CSV have null continent — these are primarily North American airports (they have `iso_region` starting with US-, CA-, etc.)

## 9. Temporal Analysis

The temporal dimension is critical for our task. Let's look at how flight activity evolves.

In [ ]:
# This plot is generated by scripts/generate_plots.py
# If not yet generated, run: python scripts/generate_plots.py
try:
    Image(filename='../results/figures/03_daily_interactions.png')
except:
    print('Plot not yet generated. Run: python scripts/generate_plots.py')

In [ ]:
try:
    Image(filename='../results/figures/04_monthly_interactions.png')
except:
    print('Plot not yet generated.')

### Interpretation

The temporal pattern reveals a **major disruption**: the COVID-19 pandemic caused a dramatic drop in flight activity starting March 2020. We expect to see:
- **Pre-COVID (2019 - early 2020)**: stable, high activity with seasonal variation
- **COVID crash (March-June 2020)**: sharp decline
- **Gradual recovery (2020-2022)**: progressive return toward pre-pandemic levels

This has important implications for temporal link prediction:
- The model must learn to handle **non-stationarity** (the distribution of interactions changes over time)
- **Train/val/test split**: if training is on 2019-mid 2021 and testing on 2022, the model trains partly on pandemic data but tests on recovery data
- **Historical baselines** may perform poorly if they weight pandemic-era data equally with pre-pandemic data

## 10. Edge Features — Aircraft Types

In [ ]:
try:
    Image(filename='../results/figures/08_top_aircraft_types.png')
except:
    print('Plot not yet generated.')

### Interpretation

The `typecode` field represents ICAO aircraft type designators:
- **B738** (Boeing 737-800): the most common aircraft, ~5.4M flights
- **A320** (Airbus A320): second most common, ~4.1M flights
- Narrow-body aircraft dominate (short/medium-haul routes)
- Wide-body codes (B77W, B763, A333) represent long-haul international routes
- **40.7% null** — many flights lack aircraft type information

For TGN, the typecode will be encoded as the edge feature (message). The high null rate means the model needs to handle missing edge features gracefully.

## 11. Degree Rank Plot (Zipf's Law)

In [ ]:
Image(filename='../results/figures/07_degree_rank.png')

### Interpretation

The rank plot (log-log) shows an approximately linear relationship, confirming a **Zipf-like (power-law) distribution**. The top airport (KORD) has ~1.9M interactions while the median airport has only ~487.

This extreme inequality means:
- **Structural heuristics** like Common Neighbors will be dominated by hubs
- **Negative sampling** must be careful — random negatives against hubs are easy, while distinguishing between two low-degree airports is harder
- The TGB uses **hist_rnd** negative sampling (historical + random) to address this

## 12. Detailed Statistics Summary

| Category | Metric | Value |
|----------|--------|------:|
| **Scale** | Total interactions | 67,169,570 |
| | Unique airports | 18,143 |
| | Unique (src, dst) pairs | 2,309,707 |
| | Avg interactions per pair | 29.1 |
| **Temporal** | Date range | 2019-01-01 to 2022-10-31 |
| | Unique timestamps | 1,385 (daily) |
| | Timestamp step | ~86,400s (1 day) |
| **Degree** | Out-degree range | [1, 1,002,075] |
| | In-degree range | [1, 918,017] |
| | Median total degree | 487 |
| | Max total degree (KORD) | 1,920,092 |
| **Structure** | Directed density | 0.0070 (0.70%) |
| | Self-loops present | Yes |
| **Node Features** | Columns | type, continent, iso_region, lon, lat |
| | Continent nulls | 49.5% (North America) |
| **Edge Features** | typecode (aircraft) | 1,124 unique, 40.7% null |
| | callsign (flight) | Unique per flight |
| **Repeated** | Pairs with 1 interaction | 1,158,643 (50.2%) |
| | Pairs with >100 | 64,358 |
| | Pairs with >1000 | 10,675 |
| | Max (KFFZ→KFFZ) | 154,371 |
| **TGB Official** | Metric | MRR |
| | Negative sampling | hist_rnd |
| | Evaluation protocol | One-vs-Many |

## 13. Key Observations for Modeling

1. **Scale-free network**: The power-law degree distribution means a few hubs dominate. The TGN's memory module should capture hub behavior differently from low-degree nodes.

2. **Heavy repetition**: 50% of pairs have only 1 interaction, but the top pairs have >100K. Historical frequency will be a strong baseline for frequent routes but useless for new ones.

3. **COVID disruption**: The pandemic creates a **distribution shift** in the data. The model needs temporal awareness to handle this.

4. **Missing features**: 40.7% of edge features (typecode) are null. We'll need a strategy for this (zero-fill, learned embedding, etc.).

5. **Self-loops**: Some airports have flights to themselves (training flights). These are valid interactions in the dataset.

6. **Geographical bias**: ADS-B coverage favors North America and Europe. The model will perform better for well-covered regions.

7. **Memory constraints**: 67M interactions don't fit in memory at once. We'll need chunked processing or sub-sampling for development, with the full dataset for final evaluation.

8. **Directed graph**: The graph is directed (src → dst), which matters for link prediction — predicting (A → B) is different from (B → A).

---

**Next**: Phase 2 — Temporal Graph Construction (`02_temporal_graph_analysis.ipynb`)